<div style="text-align: right" align="right"><i>Peter Norvig, 2018</i></div>

# Euler's Sum of Powers Conjecture

[Leonhard Euler](https://en.wikipedia.org/wiki/Leonhard_Euler) in 1769 [conjectured](https://en.wikipedia.org/wiki/Euler%27s_sum_of_powers_conjecture) that if the sum of *n* positive integer *k*th powers is itself a *k*th power, then *n* must be at least *k*. This implies:
- No solution to *a*<sup>3</sup> + *b*<sup>3</sup> = *c*<sup>3</sup> (*aka* Fermat's Last Theorem, 1637)
- No solution to *a*<sup>4</sup> + *b*<sup>4</sup> + *c*<sup>4</sup> = *d*<sup>4</sup>
- Possible solutions to *a*<sup>2</sup> + *b*<sup>2</sup> = *c*<sup>2</sup> (such as 3<sup>2</sup> + 4<sup>2</sup> = 5<sup>2</sup>)
- Possible solutions to *a*<sup>3</sup> + *b*<sup>3</sup> + *c*<sup>3</sup> = *d*<sup>3</sup> (such as 3<sup>3</sup> + 4<sup>3</sup> + 5<sup>3</sup> = 6<sup>3</sup>)

Two centuries later, in 1966, L. J. Lander and T. R. Parkin  published a famously short [article](https://fermatslibrary.com/s/counterexample-to-eulers-conjecture-on-sums-of-like-powers) giving a [counterexample](https://www.ams.org/journals/mcom/1967-21-099/S0025-5718-1967-0222008-0/S0025-5718-1967-0222008-0.pdf):

---

![](euler-paper.png)

---

# Algorithm for Finding Counterexamples

Here I sketch three algorithms that search for counterexamples, *a*<sup>5</sup> + *b*<sup>5</sup> + *c*<sup>5</sup> + *d*<sup>5</sup> = *e*<sup>5</sup>. They all enumerate values from 1 up to some maximum value *m* for various variables (Lander and Parkin chose  *m* = 250). I give the computational complexity of each:
- *O*(*m*<sup>5</sup>): Enumerate all values of *a, b, c, d, e* from 1 to *m* and check if the equation holds.
- *O*(*m*<sup>4</sup>): Enumerate all values of *a, b, c, d* from 1 to *m* and check if the sum of their fifth powers is a fifth power.
- *O*(*m*<sup>3</sup>): Precompute a dict, *pairs*, of {*a*<sup>5</sup> + *b*<sup>5</sup>: (*a*,  *b*)} entries for all 1 ≤ *a*  ≤ *b* < *m*. <br>Enumerate all values *e* from 1 to *m* and all values of *sum*<sub>1</sub> in *pairs* and check if *sum*<sub>2</sub> = *e*<sup>5</sup> - *sum*<sub>1</sub> is in *pairs*.

Let's implement the *O*(*m*<sup>3</sup>) algorithm:


In [1]:
def euler_counterexamples(m=250) -> tuple[int, int, int, int, int]:
    """Yield tuples (a, b, c, d, e) such that a^5 + b^5 + c^5 + d^5 = e^5,
    where all elements are integers, and 1 ≤ a ≤ b ≤ c ≤ d < e < m."""
    pairs  = {a**5 + b**5: (a, b) for a in range(1, m) for b in range(a, m)}
    for e in range(1, m):
        for sum1 in pairs:
            if (sum2 := e**5 - sum1) in pairs:
                abcd = sorted(pairs[sum1] + pairs[sum2])
                yield (*abcd, e)

Let's find  the first counterexample:

In [2]:
%time next(euler_counterexamples())

CPU times: user 273 ms, sys: 3.11 ms, total: 276 ms
Wall time: 275 ms


(27, 84, 110, 133, 144)

That's the same answer that Lander and Parkin got: 27<sup>5</sup> + 84<sup>5</sup> +  110<sup>5</sup> +  133<sup>5</sup> = 144<sup>5</sup> = 61,917,364,224.

(But their computation in 1966 ran a hundred thousand times slower.)

We can keep going, collecting all the counterexamples up to *m* = 1000:

In [3]:
%time set(euler_counterexamples(m=1000))

CPU times: user 38.5 s, sys: 149 ms, total: 38.7 s
Wall time: 38.7 s


{(27, 84, 110, 133, 144),
 (54, 168, 220, 266, 288),
 (81, 252, 330, 399, 432),
 (108, 336, 440, 532, 576),
 (135, 420, 550, 665, 720),
 (162, 504, 660, 798, 864)}

All the counterexamples found are multiples of the first one.

Since 1966 other mathematicians have found [other counterexamples](https://en.wikipedia.org/wiki/Euler%27s_sum_of_powers_conjecture), but all we need is one to disprove Euler's conjecture.

# Algorithm for Finding Sums of Like Powers

It is interesting to examine various values of *n* and *k*, not just *n* = 4 and and *k* = 5.

We'll use the same algorithm as above, except that instead of splitting the 4 terms into two *pairs*, we'll split the *n* terms into two *parts*, of equal size if *n* is even, or with *part*<sub>2</sub> having one more term if *n* is odd. The algorithm still enumerates over all values of *e* from 1 to *m* and over all the sums in  *part*<sub>1</sub>.

In [4]:
import itertools
import functools

def euler_sums(n: int, k: int, m=30) -> tuple[int, ...]:
    """Yield tuples of n+1 positive integers, (a_1, a_2, ... a_n, e) such that Σ a_i^k = e^k.
    m is the maximum value for any a_i or e."""
    part1 = sum_of_powers_table(n // 2, k, m)
    part2 = sum_of_powers_table(n - n // 2,  k, m)
    for e in range(1, m):
        for sum1 in part1:
            if (sum2 := e**k - sum1) in part2:
                A = sorted(part1[sum1] + part2[sum2])
                yield (*A, e)

@functools.cache
def sum_of_powers_table(n: int, k: int, m: int) -> dict[int, tuple[int, ...]]:
    """Build a table of {sum(A): A} for all tuples A = (a_1^k, ... a_n^k) with each a_i < m."""
    return {sum(a**k for a in A): A
            for A in itertools.combinations_with_replacement(range(1, m), n)}

Here are some examples:

In [5]:
set(euler_sums(n=2, k=2)) # a^2 + b^2 = c^2

{(3, 4, 5),
 (5, 12, 13),
 (6, 8, 10),
 (7, 24, 25),
 (8, 15, 17),
 (9, 12, 15),
 (10, 24, 26),
 (12, 16, 20),
 (15, 20, 25),
 (20, 21, 29)}

In [6]:
set(euler_sums(n=3, k=3)) # a^3 + b^3 + c^3 = d^3

{(1, 6, 8, 9),
 (2, 12, 16, 18),
 (3, 4, 5, 6),
 (3, 10, 18, 19),
 (3, 18, 24, 27),
 (4, 17, 22, 25),
 (6, 8, 10, 12),
 (7, 14, 17, 20),
 (9, 12, 15, 18),
 (11, 15, 27, 29),
 (12, 16, 20, 24),
 (18, 19, 21, 28)}

In [7]:
set(euler_sums(n=4, k=3)) # a^3 + b^3 + c^3 + d^3 = e^3

{(1, 1, 5, 6, 7),
 (1, 5, 7, 12, 13),
 (1, 7, 14, 14, 18),
 (1, 10, 15, 26, 28),
 (2, 2, 10, 12, 14),
 (2, 3, 8, 13, 14),
 (2, 4, 13, 27, 28),
 (2, 10, 14, 24, 26),
 (3, 3, 7, 11, 12),
 (3, 3, 15, 18, 21),
 (3, 15, 17, 21, 26),
 (4, 4, 20, 24, 28),
 (4, 6, 16, 26, 28),
 (4, 7, 8, 17, 18),
 (5, 7, 9, 10, 13),
 (5, 11, 19, 21, 26),
 (5, 14, 16, 19, 24),
 (6, 6, 14, 22, 24),
 (6, 14, 15, 18, 23),
 (7, 9, 9, 24, 25),
 (9, 13, 19, 23, 28),
 (10, 14, 18, 20, 26),
 (11, 12, 13, 14, 20)}

In [8]:
set(euler_sums(n=4, k=4, m=360)) # a^4 + b^4 + c^4 + d^4 = e^4

{(30, 120, 272, 315, 353)}

In [9]:
set(euler_sums(n=5, k=5, m=360)) # a^5 + b^5 + c^5 + d^5 + e^5 = f^5

{(7, 43, 57, 80, 100, 107),
 (14, 86, 114, 160, 200, 214),
 (19, 43, 46, 47, 67, 72),
 (21, 23, 37, 79, 84, 94),
 (21, 129, 171, 240, 300, 321),
 (38, 86, 92, 94, 134, 144),
 (42, 46, 74, 158, 168, 188),
 (57, 129, 138, 141, 201, 216),
 (63, 69, 111, 237, 252, 282),
 (76, 172, 184, 188, 268, 288)}

In [10]:
# With n = k = 6, it has been shown there are no solutions for m up to 730,000.
set(euler_sums(n=6, k=6)) 

set()